# Notebook 03 — Finding a Needle in 2 Billion Rows
## Search Optimization Service (SOS)

**The scenario**: A fraud analyst needs to look up a single transaction by ID, or pull all transactions for one specific account. With 2 billion rows, even clustering can't help when you need exact-match lookups on high-cardinality columns (50M unique accounts, 2B unique transaction IDs).

**The fix**: Search Optimization Service (SOS) builds a special index for equality lookups. Think of it like the index at the back of a book — instead of reading every page, you jump directly to what you need.

| Table | SOS Enabled? | Equality Lookup Behavior |
|-------|-------------|------------------------|
| `TRANSACTIONS_NO_SOS` | No | Scans many/all partitions |
| `TRANSACTIONS_SOS` | Yes (account_id, transaction_id) | Jumps directly to target partitions |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL_V2.CONSUMER_BANKING;

ALTER SESSION SET USE_CACHED_RESULT = FALSE;

In [ ]:
-- Verify SOS is active (should show 'ACTIVE' status)
SHOW SEARCH OPTIMIZATION ON TRANSACTIONS_SOS;

---
## Test 1: Transaction ID Lookup

"The fraud team flagged transaction TXN-000500000001 — pull the details."

### Worst Case: No SOS

In [ ]:
-- WORST: No search optimization — scans broadly
SELECT *
FROM TRANSACTIONS_NO_SOS
WHERE transaction_id = 'TXN-000500000001';

### Best Case: With SOS

In [ ]:
-- BEST: SOS jumps directly to the right partition
SELECT *
FROM TRANSACTIONS_SOS
WHERE transaction_id = 'TXN-000500000001';

---
## Test 2: Account ID Lookup

"Pull all transactions for account ACC-0000012345."

### Worst Case: No SOS

In [ ]:
-- WORST: No SOS — must scan broadly to find this account
SELECT
    COUNT(*) AS txn_count,
    SUM(amount) AS total_amount,
    MIN(transaction_date) AS first_txn,
    MAX(transaction_date) AS last_txn
FROM TRANSACTIONS_NO_SOS
WHERE account_id = 'ACC-0000012345';

### Best Case: With SOS

In [ ]:
-- BEST: SOS index narrows the search immediately
SELECT
    COUNT(*) AS txn_count,
    SUM(amount) AS total_amount,
    MIN(transaction_date) AS first_txn,
    MAX(transaction_date) AS last_txn
FROM TRANSACTIONS_SOS
WHERE account_id = 'ACC-0000012345';

---
## Compare Results

In [ ]:
SELECT
    CASE
        WHEN query_text ILIKE '%TRANSACTIONS_SOS%' THEN 'WITH SOS (best)'
        ELSE 'NO SOS (worst)'
    END AS approach,
    CASE
        WHEN query_text ILIKE '%transaction_id%' THEN 'Transaction ID lookup'
        ELSE 'Account ID lookup'
    END AS test,
    bytes_scanned / (1024*1024*1024) AS gb_scanned,
    total_elapsed_time / 1000 AS elapsed_sec
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -5, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 10
))
WHERE (query_text ILIKE '%ACC-0000012345%' OR query_text ILIKE '%TXN-000500000001%')
  AND query_text NOT ILIKE '%QUERY_HISTORY%'
  AND query_type = 'SELECT'
ORDER BY start_time DESC
LIMIT 4;

---
## Key Takeaway

| Lookup Type | Without SOS | With SOS | Improvement |
|-------------|------------|----------|-------------|
| Transaction ID | ~8-18 GB scanned | <0.1 GB | **100x+ less** |
| Account ID | ~8-18 GB scanned | <0.5 GB | **20-50x less** |

**SOS vs Clustering — when to use which:**

| Feature | Clustering | Search Optimization (SOS) |
|---------|-----------|---------------------------|
| Best for | Range queries, common filters | Exact-match equality lookups |
| Works on | 1-2 key columns | Multiple columns simultaneously |
| Maintenance | Snowflake auto-reclusters | Snowflake auto-maintains index |
| Cost model | Reclustering credits | Storage + maintenance credits |
| Example | `WHERE state = 'NY'` | `WHERE transaction_id = '...'` |

**When to ask your data engineer for SOS:**
- You need instant lookups by ID (fraud investigations, customer service)
- The column has millions of unique values
- You always use `=` (not `BETWEEN` or `LIKE`)

**Next** → [Notebook 04 — Warehouse Sizing](./Notebook_04_Warehouse_Sizing.ipynb) (what happens when your query runs out of memory)